In [ ]:
import obsidian
print(f'obsidian version: ' + obsidian.__version__)


from obsidian.experiment import AdvExpDesigner



In [ ]:
# Define continuous parameters: key -> (low, high, step)

continuous_params = {
    'temperature': (20, 80, 5),          # Linear steps of 5 between 20 and 80
    'concentration': (0.1, 1.0, 0.1),    # Linear steps of 0.1 between 0.1 and 1.0
    'pressure': (1, 16, 'geometric'),    # Geometric steps doubling from 1 to 16 (1, 2, 4, 8, 16)
    'time': (10, 1000, 'logarithmic')    # Logarithmic steps (powers of 10) between 10 and 1000
}

# Define conditional categorical parameters with subparameters and frequencies: key -> {subkey: {'freq': frequency, 'subparams': ([values], [frequencies])}}

conditional_subparameters = {
    'buffer_type': {
        'A': {'freq': 0.4, 'pH': ([6.0, 7.0, 8.0], [0.3, 0.4, 0.3])},
        'B': {'freq': 0.35, 'pH': ([5.0, 6.5], [0.7, 0.3])},
        'C': {'freq': 0.25, 'pH': ([7.5, 8.5], [0.6, 0.4])}
    },
    'catalyst': {
        'X': {'freq': 0.5, 'loading': ([0.1, 0.2, 0.3], [0.2, 0.5, 0.3])},
        'Y': {'freq': 0.3, 'loading': ([0.05, 0.15], [0.6, 0.4])},
        'Z': {'freq': 0.2, 'loading': ([0.25, 0.35], [0.7, 0.3])}
    }
}


# Initialize the designer

designer = AdvExpDesigner(continuous_params, conditional_subparameters)

In [ ]:
# Generate a design with 100 samples, optimizing categorical assignments
design = designer.generate_design(seed=123, n_samples=100, optimize_categories=True)
design

In [ ]:
# Evaluate the design quality metrics
metrics = designer.evaluate_design(design)
print("Design quality metrics:")
for metric, value in metrics.items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
# Plot histograms of all parameters and subparameters
designer.plot_histograms(design)

In [ ]:
# Plot PCA colored by 'buffer_type'
designer.plot_pca(design, hue='buffer_type')

# Plot PCA colored by 'catalyst'
designer.plot_pca(design, hue='catalyst')

In [ ]:
# Optimize design over 30 trials with 100 samples each
best_design, metrics_df = designer.optimize_design(n_trials=30, n_samples=100)

print("\nBest design metrics after optimization:")
print(metrics_df.sort_values('score', ascending=False).head(1))


In [ ]:
# Plot quality evolution over trials
designer.plot_quality_evolution(metrics_df)

In [ ]:
# Plot correlation matrix of the design

designer.plot_correlation(design)

In [ ]:
# Extend the best design by 20 new samples
extended_design, extension_summary = designer.extend_design(best_design, n=20)

print("\nExtension summary:")
print(extension_summary)

In [ ]:
# Compare empirical vs expected frequencies for categorical variables
designer.compare_frequencies(extended_design)